<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/Jup_Wateronly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt

!pip install sahi ultralytics
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

In [ ]:
def create_polygon_mask_from_points(image_path, polygon_points):
    """
    Create mask from polygon points following the shoreline
    """
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Create mask
    mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
    polygon = np.array(polygon_points, dtype=np.int32)
    cv2.fillPoly(mask, [polygon], 255)

    # Get bounding box
    x, y, w, h = cv2.boundingRect(polygon)
    water_bbox = (x, y, w, h)

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_rgb)
    axes[0].plot(polygon[:, 0], polygon[:, 1], 'magenta', linewidth=3, marker='o', markersize=4)
    axes[0].plot([polygon[-1, 0], polygon[0, 0]],
                 [polygon[-1, 1], polygon[0, 1]], 'magenta', linewidth=3)
    axes[0].fill(polygon[:, 0], polygon[:, 1], 'magenta', alpha=0.3)
    axes[0].set_title('Water Polygon - Inlet Only')
    axes[0].grid(True, alpha=0.3)

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Water Mask')

    masked = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
    axes[2].imshow(masked)
    axes[2].set_title('Result - Water Only')
    plt.tight_layout()
    plt.show()

    print(f"Water region: {water_bbox}")
    print(f"Mask coverage: {np.sum(mask > 0) / mask.size * 100:.1f}% of image")
    return mask, water_bbox

reference_image = "/content/Jupiter_Inlet/s250922w_jpg.rf.s2JZ1VgWg2GKNiAE6BEL.jpg"

# REFINED polygon - narrower left side to exclude land, keep inlet channel
water_polygon = [
    # Inlet mouth - narrower opening
    (0, 0),        # Top of inlet (moved right to exclude left land)
    (0, 0),      # Following left shoreline down
    (0, 0),
    (0, 1040),
    (950, 1050),
    (1050, 1150),
    (1100, 1150),
    (1150, 1150),
    (1300, 1200),
    (1450, 1250),
    (1580, 1260),
    (1610, 1270),
    (0, 1280),       # Narrow inlet channel
    (0, 1350),
    (1100, 1350),    # Where inlet widens
    (1250, 1450),
    (1350, 1540),
    (1400, 1620),
    (1500, 1700),    # Beach curve
    (1740, 1780),
    (1800, 1860),
    (1910, 1950),
    (2100, 2040),
    (2100, 2140),
    (2500, 2250),
    (2620, 2370),
    (2700, 2500),
    (3000, 2640),
    (3300, 3000),
    (4352, 3300),    # Bottom right
    (4352, 3264),
    (4352, 0),       # Top right corner
    (600, 0),        # Top edge (adjusted)
]

WATER_MASK, WATER_BBOX = create_polygon_mask_from_points(reference_image, water_polygon)

# Save the mask
np.save("/content/water_mask.npy", WATER_MASK)
np.save("/content/water_bbox.npy", WATER_BBOX)
np.save("/content/polygon_points.npy", np.array(water_polygon))
print("\n✓ Refined water polygon mask saved!")

In [ ]:
# Loading saved water mask
WATER_MASK = np.load("/content/water_mask.npy")

image_folder = "/content/Jupiter_Inlet"
output_folder = "/content/Jupiter_Inlet/output_sahi_yolo11_masked"
os.makedirs(output_folder, exist_ok=True)

ground_truth_json = "/content/Jupiter_Inlet/_annotations.coco.json"
with open(ground_truth_json) as f:
    gt = json.load(f)
filename_to_id = {img['file_name']: img['id'] for img in gt['images']}

# YOLOv11 model
# yolo11n.pt (fastest), yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt (most accurate)
model_path = "yolo11n.pt"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.3,
    device="cuda:0"
)

def is_in_water(bbox, mask):
    """Check if detection center point is within water mask"""
    x, y, w, h = bbox
    center_x = int(x + w/2)
    center_y = int(y + h/2)

    if 0 <= center_y < mask.shape[0] and 0 <= center_x < mask.shape[1]:
        return mask[center_y, center_x] > 0
    return False

slice_sizes = [250, 500]

for slice_size in slice_sizes:
    print(f"\n=== YOLOv11 slice {slice_size}x{slice_size} (MASKED) ===")

    predictions = []
    total_detections = 0
    filtered_detections = 0

    for filename in sorted(os.listdir(image_folder)):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(image_folder, filename)
        print(f"Processing {filename}...")

        result = get_sliced_prediction(
            image=image_path,
            detection_model=detection_model,
            slice_height=slice_size,
            slice_width=slice_size,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
        )

        for obj in result.object_prediction_list:
            if obj.category.name.lower() == "boat":
                total_detections += 1
                bbox = obj.bbox.to_xywh()

                if is_in_water(bbox, WATER_MASK):
                    filtered_detections += 1
                    predictions.append({
                        "image_id": filename_to_id[filename],
                        "category_id": 1,
                        "bbox": [float(b) for b in bbox],
                        "score": float(obj.score.value),
                    })

    pred_json_path = os.path.join(
        output_folder, f"yolo11_masked_slice_{slice_size}.json"
    )

    with open(pred_json_path, "w") as f:
        json.dump(predictions, f, indent=4)

    print(f"✓ Boats detected: {total_detections} | In water: {filtered_detections} | Filtered: {total_detections - filtered_detections}")
    print(f"✓ Saved: {pred_json_path}\n")

In [ ]:
# Paths
image_folder = "/content/Jupiter_Inlet"
gt_json = "/content/Jupiter_Inlet/_annotations.coco.json"  # or original
pred_json = "/content/Jupiter_Inlet/output_sahi_yolo11_masked/yolo11_masked_slice_250.json"
WATER_MASK = np.load("/content/water_mask.npy")

# Load annotations
with open(gt_json) as f:
    gt_data = json.load(f)

with open(pred_json) as f:
    pred_data = json.load(f)

# Map image IDs to filenames
id_to_filename = {img['id']: img['file_name'] for img in gt_data['images']}

# Organize by image
gt_by_image = {}
for ann in gt_data['annotations']:
    img_id = ann['image_id']
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann['bbox'])

pred_by_image = {}
for ann in pred_data:
    img_id = ann['image_id']
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append((ann['bbox'], ann['score']))

def draw_annotations(image_path, gt_boxes, pred_boxes, mask):
    """Draw ground truth (green) and predictions (red) with confidence scores"""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Overlay water mask (blue tint)
    mask_overlay = np.zeros_like(img)
    mask_overlay[:,:,2] = mask  # Blue channel
    img = cv2.addWeighted(img, 1.0, mask_overlay, 0.15, 0)

    # Draw ground truth (GREEN)
    for bbox in gt_boxes:
        x, y, w, h = [int(round(float(v))) for v in bbox]
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(img, 'GT', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX,
                   0.6, (0, 255, 0), 2)

    # Draw predictions (RED)
    for bbox, score in pred_boxes:
        x, y, w, h = [int(round(float(v))) for v in bbox]
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img, f'{score:.2f}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX,
                   0.5, (255, 0, 0), 2)

    # Add legend
    cv2.rectangle(img, (10, 10), (200, 90), (255, 255, 255), -1)
    cv2.putText(img, 'GREEN = Ground Truth', (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(img, 'RED = Predictions', (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    cv2.putText(img, f'GT: {len(gt_boxes)} | Pred: {len(pred_boxes)}',
               (15, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1)

    return img

# Visualize all images - display in notebook
for img_id, filename in sorted(id_to_filename.items()):
    image_path = f"{image_folder}/{filename}"

    gt_boxes = gt_by_image.get(img_id, [])
    pred_boxes = pred_by_image.get(img_id, [])

    img_with_boxes = draw_annotations(image_path, gt_boxes, pred_boxes, WATER_MASK)

    # Display in notebook
    plt.figure(figsize=(16, 12))
    plt.imshow(img_with_boxes)
    plt.axis('off')
    plt.title(f'{filename}\nGround Truth: {len(gt_boxes)} boats | Predictions: {len(pred_boxes)} boats',
             fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"{'='*60}")
    print(f"Image: {filename}")
    print(f"Ground Truth Boats: {len(gt_boxes)}")
    print(f"Predicted Boats: {len(pred_boxes)}")
    print(f"{'='*60}\n")